# Differential RV recovery (two injections)

Minimal E2E check of the differential-RV requirement:

> If two datasets have injected RVs `(rv1, rv2)` with `drv = rv2 − rv1`, then after the DRP
> `|estimated_drv − drv| < 10 cm/s`.

This notebook builds **two** BSPEC UTRs on fiber 1 with:

- different noise seeds
- injected RVs near **−50 km/s**, separated by **5 m/s**

Each run uses the packaged line-ZP WAVE path (`create_ramp_from_spectrum` paints at RV=0 and
injects via `WAVE × (1+z)`). Stage 0 → 1 → 2 recover each absolute RV; the **difference** is
what the requirement constrains.

**Environment variables** (set in your shell / kernel before running):

| Variable | Required when | Purpose |
|---|---|---|
| `KOA_CALIBRATION_CACHE` | Always | Local KOA calibration cache |
| `KECK_OBSERVER_EMAIL` | Syncing from KOA | Keck Observer email |
| `KECK_OBSERVER_PASSWORD` | Syncing from KOA | Keck Observer password |

If `KOA_CALIBRATION_CACHE` is unset, the setup cell sets it under `OUT`.


In [ ]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

# OUTPUT DIRECTORY — update this to your desired output directory
OUT = "/Users/ckilpatrick/hispec_rv_recovery_sim"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)


In [ ]:
from hispecdrp.calibrations import verify_koa_calibrations

SPECTROGRAPH = "BSPEC"
summary = verify_koa_calibrations(SPECTROGRAPH, ignore_missing_cal_types=["lfc_wavesol"])

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])


## 2. Injection configuration

Common-mode zero point near −50 km/s; differential `drv = 5 m/s`. Distinct UTR noise seeds.


In [ ]:
FIBERS = [1]

# Absolute zero point (km/s) and differential (m/s)
RV_ZERO_POINT_KM_S = -50.0
DRV_M_S = 5.0  # rv2 − rv1

RV1_KM_S = RV_ZERO_POINT_KM_S
RV2_KM_S = RV_ZERO_POINT_KM_S + DRV_M_S / 1000.0

SEED1 = 0
SEED2 = 1

REQUIREMENT_M_S = 0.10  # 10 cm/s

print(f"fiber(s): {FIBERS}")
print(f"rv1 = {RV1_KM_S:.6f} km/s  ({RV1_KM_S * 1000.0:.3f} m/s), seed={SEED1}")
print(f"rv2 = {RV2_KM_S:.6f} km/s  ({RV2_KM_S * 1000.0:.3f} m/s), seed={SEED2}")
print(f"drv = rv2 − rv1 = {DRV_M_S:.3f} m/s")
print(f"requirement: |estimated_drv − drv| < {REQUIREMENT_M_S:.2f} m/s (10 cm/s)")


## 3. Simulate two UTRs → Stage 0 / 1 / 2 → recover RVs

Each injection is written under its own subdirectory of `OUT`. Stage-1 uses the
in-memory RV-scaled packaged WAVE from `create_ramp_from_spectrum`. Traces / fiberflat
come from the local caldb (registered on the first run).


In [ ]:
from pathlib import Path

from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline
from hispecdrp.simulations.simulate_up_the_ramp import create_ramp_from_spectrum


def run_one_injection(label: str, injected_rv_km_s: float, seed: int) -> dict:
    """Build one UTR, reduce Stage0–2, return recovered aggregate RV (m/s)."""
    run_dir = str(Path(OUT) / label)
    os.makedirs(run_dir, exist_ok=True)
    injected_m_s = float(injected_rv_km_s) * 1000.0

    print(f"\n=== {label}: inject {injected_rv_km_s:.6f} km/s, seed={seed} ===")
    ramp, cals, rate_map = create_ramp_from_spectrum(
        run_dir,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename="lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5",
        mag_ab=3.05,
        filter_band="J",
        seed=seed,
        radial_velocity_km_s=injected_rv_km_s,
        barycentric_velocity_km_s=0.0,
    )
    print(
        f"ramp={ramp.meta.filename}  shape={ramp.data.shape}  "
        f"wavesol note={getattr(getattr(cals.get('wavesol'), 'meta', None), 'note', None)}"
    )

    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline(output_dir=run_dir).apply(ramp)
    l1 = Stage1Pipeline(
        output_dir=run_dir,
        prim_args={
            "spec_extract": {"do_fibers": do_fibers},
            "set_wavesol": {"wavesol": cals["wavesol"]},
        },
    ).apply(l0)
    l2 = Stage2Pipeline(
        output_dir=run_dir,
        prim_args={
            "blaze_corr": {"do_fibers": do_fibers},
            "stellar_rv": {
                "ccf_input_velocity": injected_m_s,
                "do_fibers": list(FIBERS),
                "save_preview": False,
                "preview_write_png": False,
            },
        },
    ).apply(l1)

    fib = FIBERS[0]
    avg = getattr(l2, f"rvccfavg_fib{fib}", None)
    if avg is None or len(avg) == 0:
        raise RuntimeError(f"{label}: missing rvccfavg_fib{fib}")
    rv_rec = float(avg["RV"][0])
    rv_err = float(avg["RV_ERR"][0])
    print(
        f"fiber {fib}: recovered RV = {rv_rec:.4f} ± {rv_err:.4f} m/s  "
        f"(bias vs inject = {rv_rec - injected_m_s:.4f} m/s)"
    )
    return {
        "label": label,
        "run_dir": run_dir,
        "seed": seed,
        "inject_km_s": float(injected_rv_km_s),
        "inject_m_s": injected_m_s,
        "recover_m_s": rv_rec,
        "recover_err_m_s": rv_err,
        "l2": l2,
    }


result1 = run_one_injection("inj1_rv1", RV1_KM_S, SEED1)
result2 = run_one_injection("inj2_rv2", RV2_KM_S, SEED2)


## 4. Differential RV score vs 10 cm/s requirement


In [ ]:
drv = result2["inject_m_s"] - result1["inject_m_s"]
estimated_drv = result2["recover_m_s"] - result1["recover_m_s"]
residual = estimated_drv - drv
passed = abs(residual) < REQUIREMENT_M_S

print("=== Differential RV recovery ===")
print(
    f"rv1 inject = {result1['inject_m_s']:.4f} m/s  "
    f"recover = {result1['recover_m_s']:.4f} ± {result1['recover_err_m_s']:.4f} m/s"
)
print(
    f"rv2 inject = {result2['inject_m_s']:.4f} m/s  "
    f"recover = {result2['recover_m_s']:.4f} ± {result2['recover_err_m_s']:.4f} m/s"
)
print(f"drv            = {drv:.6f} m/s")
print(f"estimated_drv  = {estimated_drv:.6f} m/s")
print(f"estimated_drv − drv = {residual:.6f} m/s  ({residual * 100.0:.3f} cm/s)")
print(f"requirement    = |estimated_drv − drv| < {REQUIREMENT_M_S:.2f} m/s (10 cm/s)")
print(f"PASS: {passed}" if passed else f"FAIL: {passed}")

assert passed, (
    f"Differential RV residual {residual:.4f} m/s exceeds "
    f"{REQUIREMENT_M_S:.2f} m/s (10 cm/s) requirement"
)
